# 01 – Time Series Clustering på NetCDF Cubes

**Formål:** Kør Time Series Clustering (TSC) på de NetCDF space-time cubes fra `00_space_time_cube_processing.ipynb`.  
For hvert af de 23 variable sammenkædes alle tidsintervaller til en 30-årig profil per lokalitet, hvorefter `TimeSeriesKMeans` (DTW, 6 klynger) køres.  

**Outputs:**
- `data/processed/tsc_netcdf_vector/{var}_tsc.gpkg` – ét GeoPackage per variabel med kolonne `cluster_tsc` (0–5)
- `results/metrics/charts_html/{var}_tsc_timeseries.html` – interaktivt Plotly-tidsseriekort
- `data/processed/tsc_netcdf_summary.csv` – opsummerende statistik

In [1]:
# ── Cell 1: Imports & konfiguration ────────────────────────────────────────────
import os
import re
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import plotly.graph_objects as go
from sklearn.metrics import silhouette_score
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance

warnings.filterwarnings('ignore')

# ── Paths ───────────────────────────────────────────────────────────────────────
REPO_ROOT   = Path('..').resolve()
NETCDF_DIR  = REPO_ROOT / 'data' / 'processed' / 'NetCDF_cubes'
VECTOR_DIR  = REPO_ROOT / 'data' / 'processed' / 'tsc_netcdf_vector'
CHARTS_DIR  = REPO_ROOT / 'results' / 'metrics' / 'charts_html'
SHAPEFILE   = REPO_ROOT / 'data' / 'raw' / 'nabolag_inkl_y_kom_shapefile' / 'cluster_outer_v1.shp'
SUMMARY_CSV = REPO_ROOT / 'data' / 'processed' / 'tsc_netcdf_summary.csv'

VECTOR_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Clustering config ───────────────────────────────────────────────────────────
N_CLUSTERS   = 6
METRIC       = 'dtw'
MAX_ITER     = 100
RANDOM_STATE = 42

# Plotly colours for clusters 0–5
CLUSTER_COLOURS = [
    '#1f77b4', '#ff7f0e', '#2ca02c',
    '#d62728', '#9467bd', '#8c564b'
]

print('Paths ok')
print(f'NetCDF dir  : {NETCDF_DIR}')
print(f'Vector dir  : {VECTOR_DIR}')
print(f'Charts dir  : {CHARTS_DIR}')
print(f'Shapefile   : {SHAPEFILE}')

c:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\.venv\Lib\site-packages\tslearn\bases\bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


Paths ok
NetCDF dir  : C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\processed\NetCDF_cubes
Vector dir  : C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\processed\tsc_netcdf_vector
Charts dir  : C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\results\metrics\charts_html
Shapefile   : C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\raw\nabolag_inkl_y_kom_shapefile\cluster_outer_v1.shp


In [2]:
# ── Cell 2: Opdag NetCDF-filer og gruppér per variabel ─────────────────────────
# Filnavnsformat: cluster_{var}_long_{start}_{end}.nc
PATTERN = re.compile(r'^cluster_(.+)_long_(\d{4})_(\d{4})\.nc$')

def discover_variable_cubes(netcdf_dir: Path) -> dict:
    """Returnerer dict {variable: [(start, end, path), ...]}, sorteret efter interval."""
    grouped = defaultdict(list)
    for f in netcdf_dir.glob('*.nc'):
        m = PATTERN.match(f.name)
        if m:
            var, start, end = m.group(1), int(m.group(2)), int(m.group(3))
            grouped[var].append((start, end, f))
    # Sort each variable's list by interval start
    return {var: sorted(paths, key=lambda x: x[0]) for var, paths in sorted(grouped.items())}

VARIABLE_CUBES = discover_variable_cubes(NETCDF_DIR)

print(f'Fundet {len(VARIABLE_CUBES)} variable:')
for var, intervals in VARIABLE_CUBES.items():
    print(f'  {var:25s}  {len(intervals)} intervaller '
          f'({intervals[0][0]}–{intervals[-1][1]})')

Fundet 20 variable:
  EMUB                       1 intervaller (1990–2020)
  PMB                        1 intervaller (1990–2020)
  PUB                        1 intervaller (1990–2020)
  age_18_25                  1 intervaller (1990–2020)
  age_26_40                  1 intervaller (1990–2020)
  age_41_55                  1 intervaller (1990–2020)
  age_56_69                  1 intervaller (1990–2020)
  crime_main_y               1 intervaller (1990–2020)
  disp_inc                   1 intervaller (1990–2020)
  emp                        1 intervaller (1990–2020)
  grund                      1 intervaller (1990–2020)
  gym_erhv                   1 intervaller (1990–2020)
  lvu                        1 intervaller (1990–2020)
  mean_price                 1 intervaller (1990–2020)
  mean_sqm                   1 intervaller (1990–2020)
  mig_in                     1 intervaller (1990–2020)
  mig_out                    1 intervaller (1990–2020)
  ool                        1 intervaller (1

In [3]:
# ── Cell 3: Hjælpefunktioner: Indlæs & sammenkæd NetCDF kuber ─────────────────

def load_and_concat_variable(var: str, intervals: list) -> tuple:
    """
    Indlæser alle NetCDF-kuber for én variabel og sammenkæder dem langs tidsaksen.
    
    Returnerer:
        location_ids  – array af cluster-IDs (str)
        time_labels   – liste af tids-labels (str)
        X             – numpy array, shape (n_locations, n_timepoints)
    """
    arrays = []
    all_time_labels = []
    location_sets = []

    for start, end, path in intervals:
        ds = xr.open_dataset(path)
        # Find den primære DataArray (ikke koordinat-variable)
        data_vars = [v for v in ds.data_vars]
        da = ds[data_vars[0]] if data_vars else ds[var]

        # Forventet dims: (location, time)
        if 'location' not in da.dims or 'time' not in da.dims:
            # Prøv generisk: antag første dim = location, anden = time
            da = da.rename({da.dims[0]: 'location', da.dims[1]: 'time'})

        loc_ids = da.coords['location'].values.astype(str)
        time_coords = da.coords['time'].values

        # Tidslabel: brug interval-streng hvis koordinater ikke er sigende
        if len(time_coords) > 0:
            try:
                labels = [str(t)[:10] for t in time_coords]
            except Exception:
                labels = [f'{start}-{end}_t{i}' for i in range(len(time_coords))]
        else:
            labels = [f'{start}-{end}']

        arrays.append((loc_ids, da.values))  # values: (n_loc, n_time)
        all_time_labels.extend(labels)
        location_sets.append(set(loc_ids))
        ds.close()

    # Fælles locations (inner join) for at undgå NaN-rækker fra manglende kuber
    common_locs = location_sets[0]
    for s in location_sets[1:]:
        common_locs = common_locs & s
    common_locs = sorted(common_locs)

    # Sammensæt matrix
    concat_parts = []
    for loc_ids, values in arrays:
        loc_idx = {l: i for i, l in enumerate(loc_ids)}
        rows = [loc_idx[l] for l in common_locs]
        concat_parts.append(values[rows, :])

    X = np.concatenate(concat_parts, axis=1)  # (n_common_locs, total_timepoints)
    return np.array(common_locs), all_time_labels, X


# Hurtig smoke-test på én variabel
_var_test = list(VARIABLE_CUBES.keys())[0]
_locs, _tlabels, _X = load_and_concat_variable(_var_test, VARIABLE_CUBES[_var_test])
print(f'Test variabel  : {_var_test}')
print(f'Locations      : {len(_locs)}')
print(f'Tidspunkter    : {len(_tlabels)}')
print(f'Matrix shape   : {_X.shape}')
print(f'NaN-andel      : {np.isnan(_X).mean():.2%}')

Test variabel  : EMUB
Locations      : 2228
Tidspunkter    : 31
Matrix shape   : (2228, 31)
NaN-andel      : 0.00%


In [4]:
# ── Cell 4: TSC clustering-funktion ───────────────────────────────────────────

def run_tsc(X: np.ndarray, location_ids: np.ndarray) -> tuple:
    """
    Kører TimeSeriesKMeans (DTW) på matrix X (n_loc × n_time).
    
    Returnerer:
        labels  – cluster-ID per lokation (int array 0–5)
        km      – fitted TimeSeriesKMeans objekt
        X_scaled – skaleret data (til silhouette-beregning)
    """
    # NaN-håndtering: ffill → bfill → 0
    X_filled = np.zeros_like(X, dtype=float)
    for i in range(X.shape[0]):
        ts = pd.Series(X[i])
        filled = ts.ffill().bfill().fillna(0).values
        X_filled[i] = filled

    # Reshape til tslearn format: (n_samples, n_timesteps, 1)
    X_3d = X_filled[:, :, np.newaxis]

    # Standardisering
    scaler = TimeSeriesScalerMeanVariance()
    X_scaled = scaler.fit_transform(X_3d)

    # Clustering
    km = TimeSeriesKMeans(
        n_clusters=N_CLUSTERS,
        metric=METRIC,
        max_iter=MAX_ITER,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0
    )
    labels = km.fit_predict(X_scaled)
    return labels, km, X_scaled


print('run_tsc() defineret.')

run_tsc() defineret.


In [5]:
# ── Cell 5: GeoPackage-eksportfunktion ─────────────────────────────────────────

def _detect_join_key(gdf: gpd.GeoDataFrame) -> str:
    """Find join-kolonne i shapefile (munic_clus → cluster_id → heuristik)."""
    for candidate in ['munic_clus', 'cluster_id', 'Cluster_id', 'CLUSTER_ID', 'cluster_ID']:
        if candidate in gdf.columns:
            return candidate
    # Heuristisk: kolonnenavn indeholder 'munic' eller 'clus'
    for col in gdf.columns:
        if 'munic' in col.lower() or 'clus' in col.lower():
            return col
    raise ValueError(f'Ingen passende join-kolonne fundet. Kolonner: {list(gdf.columns)}')


def save_gpkg(location_ids: np.ndarray, labels: np.ndarray, var: str,
              period_start: int = None, period_end: int = None) -> Path:
    """
    Merger cluster-labels til shapefile og gemmer som GeoPackage.
    Filnavn inkluderer periode-suffix hvis angivet.

    Returnerer output-stien.
    """
    gdf = gpd.read_file(SHAPEFILE)
    join_col = _detect_join_key(gdf)

    # Fjern eventuel 'fid'-kolonne der konflikter med GeoPackage's auto-FID
    fid_cols = [c for c in gdf.columns if c.lower() == 'fid']
    if fid_cols:
        gdf = gdf.drop(columns=fid_cols)

    # Normalisér IDs: strip whitespace og leading zeros for konsistens
    gdf[join_col] = gdf[join_col].astype(str).str.strip().str.lstrip('0')

    cluster_df = pd.DataFrame({
        'location': pd.Series(location_ids).astype(str).str.strip().str.lstrip('0'),
        'cluster_tsc': labels.astype(int)
    })

    gdf_merged = gdf.merge(
        cluster_df,
        left_on=join_col,
        right_on='location',
        how='left'
    )

    matched = gdf_merged['cluster_tsc'].notna().sum()
    total   = len(gdf_merged)
    print(f'  Merge: {matched}/{total} lokationer matchet')

    period_str = f'{period_start}_{period_end}' if period_start is not None else 'full'
    out_path = VECTOR_DIR / f'{var}_{period_str}_tsc.gpkg'
    gdf_merged.to_file(out_path, driver='GPKG')
    return out_path


print('save_gpkg() defineret.')


save_gpkg() defineret.


In [6]:
# ── Cell 6: Plotly-visualiseringsfunktion ──────────────────────────────────────

def plot_centroids(km: TimeSeriesKMeans, time_labels: list, var: str,
                   period_start: int = None, period_end: int = None) -> Path:
    """
    Plotter 6 cluster-centroider som interaktive Plotly-linjer og gemmer HTML.
    Filnavn og titel inkluderer periode-suffix hvis angivet.

    Centroids shape: (n_clusters, n_timesteps, 1) → squeeze til (n_clusters, n_timesteps)
    """
    centroids = km.cluster_centers_[:, :, 0]  # (6, n_time)
    n_time = centroids.shape[1]

    # Brug kortere time_labels hvis de er for lange
    if len(time_labels) == n_time:
        x_labels = time_labels
    else:
        x_labels = list(range(n_time))

    period_str   = f'{period_start}_{period_end}' if period_start is not None else 'full'
    title_period = f' ({period_start}–{period_end})' if period_start is not None else ''

    fig = go.Figure()
    for c in range(N_CLUSTERS):
        fig.add_trace(go.Scatter(
            x=x_labels,
            y=centroids[c],
            mode='lines+markers',
            name=f'Cluster {c}',
            line=dict(color=CLUSTER_COLOURS[c], width=2),
            marker=dict(size=5)
        ))

    fig.update_layout(
        title=dict(
            text=f'TSC Centroider – {var}{title_period}',
            font=dict(size=18)
        ),
        xaxis_title='Tidspunkt',
        yaxis_title='Normaliseret værdi (z-score)',
        legend_title='Cluster',
        hovermode='x unified',
        template='plotly_white',
        height=500
    )

    out_path = CHARTS_DIR / f'{var}_{period_str}_tsc_timeseries.html'
    fig.write_html(str(out_path))
    return out_path


print('plot_centroids() defineret.')


plot_centroids() defineret.


In [7]:
# ── Cell 7: Summary-funktion ───────────────────────────────────────────────────

def compute_summary(var: str, labels: np.ndarray, km: TimeSeriesKMeans,
                    X_scaled: np.ndarray, n_intervals: int, n_timepoints: int) -> dict:
    """
    Beregner opsummerende statistik for én TSC-kørsel.
    Silhouette-score beregnes med Euclidean på fladtrykt matrix (hurtig approksimation).
    """
    cluster_sizes = pd.Series(labels).value_counts().sort_index()
    sizes_str = ', '.join(f'C{k}={v}' for k, v in cluster_sizes.items())

    # Silhouette (Euclidean approx på flattened 2D)
    X_flat = X_scaled[:, :, 0]  # (n_loc, n_time)
    try:
        sil = silhouette_score(X_flat, labels, metric='euclidean')
    except Exception:
        sil = np.nan

    return {
        'variable':     var,
        'n_locations':  len(labels),
        'n_intervals':  n_intervals,
        'n_timepoints': n_timepoints,
        'inertia':      round(km.inertia_, 4),
        'silhouette':   round(sil, 4) if not np.isnan(sil) else None,
        'cluster_sizes': sizes_str
    }


print('compute_summary() defineret.')

compute_summary() defineret.


In [8]:
# ── Cell 8: Hoved-batch loop ───────────────────────────────────────────────────
# Kører TSC på hvert interval separat for alle 23 variable (23 × 3 = 69 kørsler).

summary_rows = []
failed_vars  = []

total_runs = sum(len(ivs) for ivs in VARIABLE_CUBES.values())
run_num = 0

for var, intervals in VARIABLE_CUBES.items():
    for (start, end, path) in intervals:
        run_num += 1
        print(f'\n[{run_num}/{total_runs}] {var}  {start}–{end} ...')

        try:
            # 1. Indlæs ét interval
            location_ids, time_labels, X = load_and_concat_variable(var, [(start, end, path)])
            print(f'  Indlæst  : {X.shape[0]} lokationer × {X.shape[1]} tidspunkter')

            # 2. Kør TSC
            labels, km, X_scaled = run_tsc(X, location_ids)
            unique, counts = np.unique(labels, return_counts=True)
            print(f'  Clustered: {dict(zip(unique.tolist(), counts.tolist()))}')

            # 3. Gem GeoPackage
            gpkg_path = save_gpkg(location_ids, labels, var, start, end)
            print(f'  Gemt gpkg: {gpkg_path.name}')

            # 4. Plot og gem chart
            chart_path = plot_centroids(km, time_labels, var, start, end)
            print(f'  Gemt html: {chart_path.name}')

            # 5. Summary
            row = compute_summary(var, labels, km, X_scaled, 1, X.shape[1])
            row['period_start'] = start
            row['period_end']   = end
            summary_rows.append(row)
            print(f'  Silhouette={row["silhouette"]}, Inertia={row["inertia"]}')

        except Exception as e:
            print(f'  FEJL: {e}')
            failed_vars.append((f'{var} {start}-{end}', str(e)))

print(f'\n{"="*60}')
print(f'Færdig: {len(summary_rows)}/{total_runs} kørsler gennemført.')
if failed_vars:
    print('Fejlede:')
    for v, err in failed_vars:
        print(f'  {v}: {err}')



[1/20] EMUB  1990–2020 ...
  Indlæst  : 2228 lokationer × 31 tidspunkter
  Clustered: {0: 295, 1: 407, 2: 398, 3: 385, 4: 333, 5: 410}
  Merge: 2228/2232 lokationer matchet
  Gemt gpkg: EMUB_1990_2020_tsc.gpkg
  Gemt html: EMUB_1990_2020_tsc_timeseries.html
  Silhouette=0.0601, Inertia=6.8319

[2/20] PMB  1990–2020 ...
  Indlæst  : 2228 lokationer × 31 tidspunkter
  Clustered: {0: 396, 1: 608, 2: 471, 3: 249, 4: 330, 5: 174}
  Merge: 2228/2232 lokationer matchet
  Gemt gpkg: PMB_1990_2020_tsc.gpkg
  Gemt html: PMB_1990_2020_tsc_timeseries.html
  Silhouette=0.0556, Inertia=5.7371

[3/20] PUB  1990–2020 ...
  Indlæst  : 2228 lokationer × 31 tidspunkter
  Clustered: {0: 339, 1: 554, 2: 335, 3: 239, 4: 297, 5: 464}
  Merge: 2228/2232 lokationer matchet
  Gemt gpkg: PUB_1990_2020_tsc.gpkg
  Gemt html: PUB_1990_2020_tsc_timeseries.html
  Silhouette=0.0448, Inertia=6.0674

[4/20] age_18_25  1990–2020 ...
  Indlæst  : 2228 lokationer × 31 tidspunkter
  Clustered: {0: 392, 1: 392, 2: 416, 3: 3

In [9]:
# ── Cell 9: Gem summary CSV og vis resultat ────────────────────────────────────

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)

print(f'Summary gemt: {SUMMARY_CSV}')
print(f'\nFiler i tsc_netcdf_vector/:')
gpkg_files = sorted(VECTOR_DIR.glob('*.gpkg'))
for f in gpkg_files:
    print(f'  {f.name}')

print(f'\nFiler i charts_html/ (tsc):')
chart_files = sorted(CHARTS_DIR.glob('*_tsc_timeseries.html'))
for f in chart_files:
    print(f'  {f.name}')

print(f'\nSummary tabel:')
cols = ['variable', 'period_start', 'period_end', 'n_locations',
        'n_timepoints', 'inertia', 'silhouette', 'cluster_sizes']
display(summary_df[[c for c in cols if c in summary_df.columns]])


Summary gemt: C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\processed\tsc_netcdf_summary.csv

Filer i tsc_netcdf_vector/:
  age_18_25_1990_2020_tsc.gpkg
  age_26_40_1990_2020_tsc.gpkg
  age_41_55_1990_2020_tsc.gpkg
  age_56_69_1990_2020_tsc.gpkg
  crime_main_y_1990_2020_tsc.gpkg
  disp_inc_1990_2020_tsc.gpkg
  emp_1990_2020_tsc.gpkg
  EMUB_1990_2020_tsc.gpkg
  grund_1990_2020_tsc.gpkg
  gym_erhv_1990_2020_tsc.gpkg
  lvu_1990_2020_tsc.gpkg
  mean_price_1990_2020_tsc.gpkg
  mean_sqm_1990_2020_tsc.gpkg
  mig_in_1990_2020_tsc.gpkg
  mig_out_1990_2020_tsc.gpkg
  ool_1990_2020_tsc.gpkg
  PMB_1990_2020_tsc.gpkg
  PUB_1990_2020_tsc.gpkg
  public_housing_1990_2020_tsc.gpkg
  unemp_1990_2020_tsc.gpkg

Filer i charts_html/ (tsc):
  age_18_25_1990_2020_tsc_timeseries.html
  age_26_40_1990_2020_tsc_timeseries.html
  age_41_55_1990_2020_tsc_timeseries.html
  age_56_69_1990_2020_tsc_timeseries.html
  crime_main_y_1990_2020_tsc_timeseries.html
  disp_inc_1990_2020_tsc_timeseries.

,variable,period_start,period_end,n_locations,n_timepoints,inertia,silhouette,cluster_sizes
0,EMUB,1990,2020,2228,31,6.8319,0.0601,"C0=295, C1=407, C2=398, C3=385, C4=333, C5=410"
1,PMB,1990,2020,2228,31,5.7371,0.0556,"C0=396, C1=608, C2=471, C3=249, C4=330, C5=174"
2,PUB,1990,2020,2228,31,6.0674,0.0448,"C0=339, C1=554, C2=335, C3=239, C4=297, C5=464"
3,age_18_25,1990,2020,2228,31,6.9233,0.0367,"C0=392, C1=392, C2=416, C3=396, C4=259, C5=373"
4,age_26_40,1990,2020,2228,31,5.2266,0.1001,"C0=454, C1=200, C2=472, C3=433, C4=438, C5=231"
5,age_41_55,1990,2020,2228,31,5.4533,0.0899,"C0=468, C1=367, C2=448, C3=427, C4=302, C5=216"
6,age_56_69,1990,2020,2228,31,4.1472,0.1450,"C0=561, C1=202, C2=369, C3=415, C4=419, C5=262"
7,crime_main_y,1990,2020,2228,31,9.1790,0.0295,"C0=484, C1=568, C2=46, C3=546, C4=339, C5=245"
8,disp_inc,1990,2020,2228,31,0.7139,0.0484,"C0=517, C1=919, C2=57, C3=42, C4=300, C5=393"
9,emp,1990,2020,2228,31,6.0569,0.0548,"C0=262, C1=632, C2=331, C3=400, C4=430, C5=173"


In [10]:
# ── Cell 10 (valgfri): Vis én chart inline for visuel verifikation ─────────────

PREVIEW_VAR = 'disp_inc'  # Skift til en anden variabel efter ønske

if PREVIEW_VAR in VARIABLE_CUBES:
    _locs, _tlabels, _X = load_and_concat_variable(PREVIEW_VAR, VARIABLE_CUBES[PREVIEW_VAR])
    _labels, _km, _X_scaled = run_tsc(_X, _locs)
    _centroids = _km.cluster_centers_[:, :, 0]

    fig_preview = go.Figure()
    for c in range(N_CLUSTERS):
        n_in_cluster = int((_labels == c).sum())
        fig_preview.add_trace(go.Scatter(
            x=list(range(len(_tlabels))),
            y=_centroids[c],
            mode='lines+markers',
            name=f'Cluster {c} (n={n_in_cluster})',
            line=dict(color=CLUSTER_COLOURS[c], width=2),
            marker=dict(size=5)
        ))
    fig_preview.update_layout(
        title=f'Preview: TSC centroider – {PREVIEW_VAR}',
        xaxis_title='Tidspunkt (indeks)',
        yaxis_title='Normaliseret z-score',
        template='plotly_white',
        height=500
    )
    fig_preview.show()
else:
    print(f'{PREVIEW_VAR} ikke fundet i VARIABLE_CUBES.')

In [11]:
# ── Cell 11: Verificér eksisterende GeoPackage-filer ──────────────────────────
import geopandas as gpd
from pathlib import Path

VECTOR_DIR_CHECK = Path('..') / 'data' / 'processed' / 'tsc_netcdf_vector'

gpkg_files = sorted(VECTOR_DIR_CHECK.glob('*.gpkg'))
print(f'Fandt {len(gpkg_files)} .gpkg filer i {VECTOR_DIR_CHECK}\n')
print(f'{"Fil":<50} {"Rækker":>7} {"Geom-type":<20} {"Null-geom":>9} {"CRS"}')
print('-' * 110)

problems = []
for f in gpkg_files:
    try:
        gdf = gpd.read_file(f)
        n_rows     = len(gdf)
        null_geom  = gdf.geometry.isna().sum() if gdf.geometry is not None else n_rows
        geom_types = gdf.geometry.geom_type.dropna().unique().tolist() if gdf.geometry is not None else ['INGEN']
        crs        = str(gdf.crs) if gdf.crs else 'INGEN CRS'
        status     = '⚠️ NULL-geom' if null_geom == n_rows else ('⚠️ delvis null' if null_geom > 0 else 'OK')
        print(f'{f.name:<50} {n_rows:>7} {str(geom_types):<20} {null_geom:>9}  {crs}  [{status}]')
        if null_geom > 0 or gdf.crs is None:
            problems.append(f.name)
    except Exception as e:
        print(f'{f.name:<50} FEJL: {e}')
        problems.append(f.name)

print()
if problems:
    print(f'⚠️  {len(problems)} filer med problemer:')
    for p in problems:
        print(f'   {p}')
else:
    print('✅ Alle filer ser korrekte ud.')


Fandt 20 .gpkg filer i ..\data\processed\tsc_netcdf_vector

Fil                                                 Rækker Geom-type            Null-geom CRS
--------------------------------------------------------------------------------------------------------------
age_18_25_1990_2020_tsc.gpkg                          2232 ['Polygon']                  0  EPSG:25832  [OK]
age_26_40_1990_2020_tsc.gpkg                          2232 ['Polygon']                  0  EPSG:25832  [OK]
age_41_55_1990_2020_tsc.gpkg                          2232 ['Polygon']                  0  EPSG:25832  [OK]
age_56_69_1990_2020_tsc.gpkg                          2232 ['Polygon']                  0  EPSG:25832  [OK]
crime_main_y_1990_2020_tsc.gpkg                       2232 ['Polygon']                  0  EPSG:25832  [OK]
disp_inc_1990_2020_tsc.gpkg                           2232 ['Polygon']                  0  EPSG:25832  [OK]
emp_1990_2020_tsc.gpkg                                2232 ['Polygon']                 